# Demand-Based Dynamic Pricing Models for Travel Markets

---

### The following information:

**Client:** A leading B2B tour operator in Kazakhstan and Central Asia, cooperating with 5,000+ travel agencies across destinations including Vietnam, Thailand, Egypt, Turkey, UAE and the Maldives

**Problem statement & Importance:**
Tour operators in B2B travel markets must commit to flight and hotel inventory 6-8 months in advance, without knowing the actual demand. Traditional demand estimation methods rely on historical bookings and expert judgment, which becomes insufficient when market conditions shift due to competitor pricing, seasonal search trends, and external events. This uncertainty leads to either unsold seats or missed revenue opportunities. A critical business challenge is not only whether demand can be predicted accurately - but how early in the booking window a reliable prediction can be made, giving the operator enough time to act.

**Goal:**
To develop and evaluate machine learning models that predict flight load factor from Kazakhstan to Vietnam routes (the company's most popular destination) at multiple booking windows (D-90, D-60, D-30, D-7), and to assess whether external market signals improve prediction accuracy beyond internal booking data alone.

Load Factor = Seats Sold / Total Seats x 100%

**What is a Price Revision in Travel?**
A price revision is a price correction applied by a tour operator to the base cost of a tour package. Since charter flights and hotels are contracted months in advance at a fixed cost, the operator sets an initial package price based on estimated demand. As the departure date approaches, the operator reviews actual booking pace and remaining seat availability. The revision amount is calculated on the air ticket cost but applied to the total tour package price (flight + hotel + services).

**Business Benefits:**
1. Identifies the earliest reliable prediction window for each route
2. Reduces financial risk from unsold charter seats
3. Provides a data-driven foundation for dynamic price revision decisions
4. Competitive advantage over operators using rule-based pricing only

**Relevant data collected from:**

Internal signals:
- Historical booking claims (2024-2026): cancellations, lead time, pax, hotel & flight costs
- Historical flight capacity (2024-2026): seats sold, empty seats, load factor per window

External signals at specific points in the past:
- Google Trends API: search interest per destination
- Google News API: news per country
- Competitor websites: price monitoring
- Public holiday calendars: event & holiday flags

**Approach:** A time-series approach is used throughout this project. Features are constructed at each booking window using only information available at that point in time - preventing data leakage and ensuring realistic future deployment.

**Core Task: Load Factor Prediction at multiple booking windows**

| Window | Days Before Departure | Business Meaning |
|--------|----------------------|------------------|
| D-90 | 90 days | Early signal - low booking activity |
| D-60 | 60 days | Search trends become informative |
| D-30 | 30 days | Majority of bookings visible |
| D-7  | 7 days  | Near-final occupancy picture |

For each window two experiments are conducted:
- Baseline model: internal features only (Linear Regression)
- Extended model: internal + external signals (Random Forest, XGBoost, LightGBM)

**Academic Foundation:**
Spak et al. (2025) demonstrated that gradient boosting models using external price signals achieved ~10pp MAE per flight, outperforming internal-only models by 2.5 percentage points. This study extends this approach to B2B charter tourism in Central Asia - a context not previously studied in the literature.

This project will be approached as a two-step pipeline:

Step 1 - Load Factor Prediction (regression task): Predict the Load Factor (% of seats sold) per Vietnam flight at multiple booking windows using:
- Baseline model: internal features only (Linear Regression)
- Advanced models: Random Forest, XGBoost, LightGBM

Step 2 - Price Revision Calculation (rule-based): Based on predicted load factor, the tour operator can calculate the recommended total price revision for the full tour package

# 1. Importing the Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Settings
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('All libraries loaded successfully.')

# 2. Import the Data

In [ ]:
# Load claims dataset
df_claims = pd.read_csv('data/_dp_claims_features.csv.gz', compression='gzip')
print(f'Claims shape: {df_claims.shape}')
df_claims.head(3)

In [ ]:
# Load flight load dataset
df_flights = pd.read_csv('data/_dp_flight_load_history.csv')
print(f'Flights shape: {df_flights.shape}')
df_flights.head(3)